# agg_instance notebook walkthrough (dataAdapter datasets)

This walkthrough mirrors the four helper steps exported by `revisions.new_src.agg_instance` while relying on the shared `dataAdapter` loader so you can point the same cells at any supported TU/BA benchmark.


## 0. Configure dataset + import helpers

Change `DATASET_NAME` to another entry from `revisions.new_src.dataAdapter.SUPPORTED_DATASETS` to reuse the exact same flow on a different benchmark. The cell below wires up every helper used later in the notebook.


In [ ]:
import torch

from revisions.new_src.dataAdapter import load_dataset
from revisions.new_src.explainee import fit_explainee
from revisions.new_src.agg_instance import (
    build_explainer,
    aggregate_instance_explanations,
    run_eval_summary,
    plot_eval,
)
from revisions.new_src.graph_level_dist import spectral_dist
from revisions.new_src.simgnn import SimGNN

DATASET_NAME = "MUTAG"  # swap this for any supported dataset
DATA_ROOT = "data"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = load_dataset(DATASET_NAME, root=DATA_ROOT)
observed_by_class = dataset.split_by_class()
print(f"Loaded {len(dataset)} graphs from {DATASET_NAME} across {len(observed_by_class)} classes")
print("Class histogram:", {idx: len(split) for idx, split in enumerate(observed_by_class)})


## 1. Train the explainee and configure `torch_geometric.explain.Explainer`

`fit_explainee` trains the shared `GCNClassifier` on whichever dataset you selected. Afterwards `build_explainer` wraps the explainee with the probability adapter and lets you pick any algorithm registered inside `agg_instance` (swap `algorithm` / `algorithm_kwargs` as needed).


In [ ]:
explainee, training_stats = fit_explainee(
    DATASET_NAME,
    root=DATA_ROOT,
    hidden=64,
    layers=3,
    dropout=0.2,
    epochs=30,
    batch_size=64,
    lr=1e-3,
    device=DEVICE,
)

explainer = build_explainer(
    explainee,
    algorithm="gnnexplainer",  # try "pgexplainer", Captum variants, etc.
    algorithm_kwargs={"epochs": 50},
    explanation_type="model",
    node_mask_type="object",
    edge_mask_type="object",
)

explainer


## 2. Aggregate instance explanations into motif batches

`aggregate_instance_explanations` iterates over the dataset, explains each graph, and groups the salient substructures per predicted class. Select whichever aggregation strategy best matches your study (`"wl_topk"`, `"node_threshold"`, or a custom callable).


In [ ]:
aggregation = aggregate_instance_explanations(
    dataset,
    explainer,
    explainee=explainee,
    strategy="wl_topk",
    strategy_kwargs={"wl_hops": 2, "top_p": 0.25, "min_edges": 4},
)

if not aggregation.by_class:
    raise RuntimeError("No motifs detected. Relax the aggregation thresholds or confirm the explainee's accuracy.")

for cls, batch in aggregation.by_class.items():
    print(f"Class {cls}: {batch.num_graphs} motifs, {batch.num_nodes} nodes, {batch.num_edges} edges total")


## 3. Compute the evaluation summary metrics

`run_eval_summary` drops the aggregated motifs plus observed graphs straight into `revisions.new_src.eval.eval_summary`. Here we reuse the ready-made `spectral_dist` module as a graph-level distance, but you can plug in `mcs_soft_graph_dist`, `neural_approx_ged_dist`, or any custom scorer that exposes the same API.


In [ ]:
if len(observed_by_class) < 2:
    raise ValueError("The selected dataset must expose at least two graph classes for eval_summary.")

dist_to_0 = spectral_dist(observed_by_class[0], which="laplacian", k=8)
dist_to_1 = spectral_dist(observed_by_class[1], which="laplacian", k=8)

summary_score = run_eval_summary(
    explainee,
    aggregation,
    observed_class_0=observed_by_class[0],
    observed_class_1=observed_by_class[1],
    dist_to_0=dist_to_0,
    dist_to_1=dist_to_1,
)

print(f"Eval summary score: {summary_score:.4f}")


## 4. Plot generated vs observed graphs

`plot_eval` visualises motif batches next to observed graphs while querying a GED approximator. The cell instantiates the shared `SimGNN` architecture so you can immediately load your own checkpoint via `load_state_dict` if you already trained a GED model for the dataset.


In [ ]:
sample_graph = dataset[0]
if getattr(sample_graph, 'x', None) is not None:
    node_feat_dim = sample_graph.x.size(-1) if sample_graph.x.dim() > 1 else 1
else:
    node_feat_dim = max(1, len(getattr(dataset, 'NODE_CLS', {})))

ged_model = SimGNN(
    in_dim=node_feat_dim,
    hidden_dim=64,
    hist_bins=16,
    dropout=0.1,
    use_tensor=False,
    tensor_channels=8,
)

plot_eval(
    explainee,
    aggregation,
    observed_class_0=observed_by_class[0],
    observed_class_1=observed_by_class[1],
    ged_model=ged_model,
    dataset=dataset,
    max_pairs=3,
)
